In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql import functions as F

In [0]:
df = spark.read.table('fintech360.business_transformation.bureau_data')
df.limit(5).display()

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# 1. CONFIGURATION
# ============================================================
source_table = "fintech360.business_transformation.bureau_data"
target_table = "fintech360.gold.bureau_data"

spark.sql("CREATE SCHEMA IF NOT EXISTS fintech360.gold")

# ============================================================
# 2. READ BUSINESS-TRANSFORMED DATA
# ============================================================
df = spark.read.table(source_table)

# ============================================================
# 3. DEDUPLICATE THE INCOMING BATCH
# Keep the latest transformed record per bureau record
# ============================================================
dedupe_window = Window.partitionBy("bureau_record_id").orderBy(
    F.col("business_transformed_at").desc_nulls_last(),
    F.col("updated_at").cast("long").desc_nulls_last()
)

df = (
    df.withColumn("_row_num", F.row_number().over(dedupe_window))
      .filter(F.col("_row_num") == 1)
      .drop("_row_num")
)

# ============================================================
# 4. STANDARDIZE CORE BUREAU FIELDS
# ============================================================
df = (
    df
    .withColumn("gold_bureau_record_id", F.col("bureau_record_id").cast("string"))
    .withColumn("gold_customer_id", F.col("customer_id").cast("long"))
    .withColumn("gold_bureau_provider", F.upper(F.trim(F.col("business_bureau_provider"))))
    .withColumn("gold_credit_score", F.col("business_credit_score").cast("int"))
    .withColumn("gold_score_band", F.upper(F.trim(F.col("business_score_band"))))
    .withColumn("gold_active_accounts", F.col("business_active_accounts").cast("int"))
    .withColumn("gold_total_accounts", F.col("business_total_accounts").cast("int"))
    .withColumn("gold_outstanding_amount", F.col("business_outstanding_amount").cast("decimal(18,2)"))
    .withColumn("gold_overdue_amount", F.col("business_overdue_amount").cast("decimal(18,2)"))
    .withColumn("gold_enquiries_30d", F.col("business_enquiries_30d").cast("int"))
    .withColumn("gold_enquiries_90d", F.col("business_enquiries_90d").cast("int"))
    .withColumn("gold_delinquency_ever", F.col("business_delinquency_ever").cast("int"))
    .withColumn("gold_written_off_accounts", F.col("business_written_off_accounts").cast("int"))
)

# ============================================================
# 5. REPORT DATE AND RECENCY
# ============================================================
df = (
    df
    .withColumn(
        "gold_report_timestamp",
        F.coalesce(
            F.to_timestamp(F.col("business_report_datetime")),
            F.to_timestamp(F.from_unixtime(F.col("report_ts").cast("long")))
        )
    )
    .withColumn("gold_report_date", F.to_date("gold_report_timestamp"))
    .withColumn("gold_report_month", F.date_format("gold_report_timestamp", "yyyy-MM"))
    .withColumn("gold_report_year", F.year("gold_report_timestamp"))
    .withColumn(
        "gold_report_age_days",
        F.datediff(F.current_date(), F.to_date("gold_report_timestamp"))
    )
    .withColumn(
        "gold_report_recency_segment",
        F.when(F.col("gold_report_age_days").isNull(), "UNKNOWN")
         .when(F.col("gold_report_age_days") < 0, "FUTURE_REPORT_DATE_REVIEW")
         .when(F.col("gold_report_age_days") <= 30, "0_TO_30_DAYS")
         .when(F.col("gold_report_age_days") <= 90, "31_TO_90_DAYS")
         .when(F.col("gold_report_age_days") <= 180, "91_TO_180_DAYS")
         .otherwise("OLDER_THAN_180_DAYS")
    )
)

# ============================================================
# 6. CREDIT SCORE SEGMENTS
# Thresholds are configurable business segmentation bands.
# ============================================================
df = (
    df
    .withColumn(
        "gold_credit_score_segment",
        F.when(F.col("gold_credit_score").isNull(), "UNKNOWN")
         .when(F.col("gold_credit_score") < 600, "BELOW_600")
         .when(F.col("gold_credit_score") < 650, "600_TO_649")
         .when(F.col("gold_credit_score") < 700, "650_TO_699")
         .when(F.col("gold_credit_score") < 750, "700_TO_749")
         .when(F.col("gold_credit_score") < 800, "750_TO_799")
         .otherwise("800_PLUS")
    )
    .withColumn(
        "gold_credit_profile",
        F.when(F.col("gold_credit_score").isNull(), "UNKNOWN")
         .when(F.col("gold_credit_score") < 650, "LOW_SCORE_REVIEW")
         .when(F.col("gold_credit_score") < 750, "MID_SCORE")
         .otherwise("HIGH_SCORE")
    )
)

# ============================================================
# 7. ACCOUNT UTILIZATION AND DEBT BURDEN
# ============================================================
df = (
    df
    .withColumn(
        "gold_inactive_accounts",
        F.greatest(
            F.coalesce(F.col("gold_total_accounts"), F.lit(0)) -
            F.coalesce(F.col("gold_active_accounts"), F.lit(0)),
            F.lit(0)
        )
    )
    .withColumn(
        "gold_active_account_ratio_pct",
        F.when(
            F.col("gold_total_accounts") > 0,
            F.round(
                F.col("gold_active_accounts") * 100.0 /
                F.col("gold_total_accounts"),
                2
            )
        )
    )
    .withColumn(
        "gold_overdue_to_outstanding_pct",
        F.when(
            F.col("gold_outstanding_amount") > 0,
            F.round(
                F.col("gold_overdue_amount") * 100.0 /
                F.col("gold_outstanding_amount"),
                2
            )
        )
    )
    .withColumn(
        "gold_has_outstanding_debt_flag",
        F.when(F.col("gold_outstanding_amount") > 0, 1).otherwise(0)
    )
    .withColumn(
        "gold_has_overdue_amount_flag",
        F.when(F.col("gold_overdue_amount") > 0, 1).otherwise(0)
    )
    .withColumn(
        "gold_outstanding_amount_band",
        F.when(F.col("gold_outstanding_amount").isNull(), "UNKNOWN")
         .when(F.col("gold_outstanding_amount") <= 0, "NO_OUTSTANDING")
         .when(F.col("gold_outstanding_amount") < 10000, "BELOW_10K")
         .when(F.col("gold_outstanding_amount") < 25000, "10K_TO_24999")
         .when(F.col("gold_outstanding_amount") < 50000, "25K_TO_49999")
         .otherwise("50K_PLUS")
    )
    .withColumn(
        "gold_overdue_amount_band",
        F.when(F.col("gold_overdue_amount").isNull(), "UNKNOWN")
         .when(F.col("gold_overdue_amount") <= 0, "NO_OVERDUE")
         .when(F.col("gold_overdue_amount") < 500, "BELOW_500")
         .when(F.col("gold_overdue_amount") < 2000, "500_TO_1999")
         .when(F.col("gold_overdue_amount") < 5000, "2000_TO_4999")
         .otherwise("5000_PLUS")
    )
)

# ============================================================
# 8. ENQUIRY ACTIVITY
# ============================================================
df = (
    df
    .withColumn(
        "gold_enquiries_31_to_90d",
        F.greatest(
            F.coalesce(F.col("gold_enquiries_90d"), F.lit(0)) -
            F.coalesce(F.col("gold_enquiries_30d"), F.lit(0)),
            F.lit(0)
        )
    )
    .withColumn(
        "gold_enquiry_intensity_30d",
        F.when(F.col("gold_enquiries_30d").isNull(), "UNKNOWN")
         .when(F.col("gold_enquiries_30d") == 0, "NO_RECENT_ENQUIRIES")
         .when(F.col("gold_enquiries_30d") <= 2, "LOW_ENQUIRY_ACTIVITY")
         .when(F.col("gold_enquiries_30d") <= 5, "MODERATE_ENQUIRY_ACTIVITY")
         .otherwise("HIGH_ENQUIRY_ACTIVITY")
    )
    .withColumn(
        "gold_enquiry_intensity_90d",
        F.when(F.col("gold_enquiries_90d").isNull(), "UNKNOWN")
         .when(F.col("gold_enquiries_90d") <= 2, "LOW_ENQUIRY_ACTIVITY")
         .when(F.col("gold_enquiries_90d") <= 5, "MODERATE_ENQUIRY_ACTIVITY")
         .otherwise("HIGH_ENQUIRY_ACTIVITY")
    )
)

# ============================================================
# 9. DELINQUENCY AND WRITE-OFF SIGNALS
# ============================================================
df = (
    df
    .withColumn(
        "gold_historical_delinquency_flag",
        F.when(F.col("gold_delinquency_ever") == 1, 1).otherwise(0)
    )
    .withColumn(
        "gold_written_off_account_flag",
        F.when(F.col("gold_written_off_accounts") > 0, 1).otherwise(0)
    )
    .withColumn(
        "gold_debt_review_signal",
        F.when(F.col("gold_overdue_amount").isNull(), "OVERDUE_DATA_MISSING")
         .when(F.col("gold_overdue_amount") < 0, "INVALID_OVERDUE_AMOUNT")
         .when(F.col("gold_overdue_amount") > 0, "OVERDUE_EXPOSURE_REVIEW")
         .otherwise("NO_CURRENT_OVERDUE_REPORTED")
    )
)

# ============================================================
# 10. OVERALL REVIEW SIGNAL
# This is a screening signal, not an automated credit decision.
# ============================================================
df = (
    df
    .withColumn(
        "gold_review_signal",
        F.when(F.col("gold_bureau_record_id").isNull(), "MISSING_BUREAU_RECORD_ID")
         .when(F.col("gold_customer_id").isNull(), "MISSING_CUSTOMER_ID")
         .when(F.col("gold_credit_score").isNull(), "MISSING_CREDIT_SCORE")
         .when(
             (F.col("gold_credit_score") < 300) |
             (F.col("gold_credit_score") > 900),
             "CREDIT_SCORE_OUT_OF_RANGE"
         )
         .when(
             F.col("gold_active_accounts").isNotNull() &
             F.col("gold_total_accounts").isNotNull() &
             (
                 (F.col("gold_active_accounts") < 0) |
                 (F.col("gold_total_accounts") < 0) |
                 (F.col("gold_active_accounts") > F.col("gold_total_accounts"))
             ),
             "INVALID_ACCOUNT_COUNTS"
         )
         .when(
             F.col("gold_outstanding_amount").isNull() |
             F.col("gold_overdue_amount").isNull(),
             "MISSING_DEBT_DATA"
         )
         .when(
             (F.col("gold_outstanding_amount") < 0) |
             (F.col("gold_overdue_amount") < 0),
             "INVALID_DEBT_AMOUNT"
         )
         .when(F.col("gold_written_off_account_flag") == 1, "WRITTEN_OFF_ACCOUNT_REVIEW")
         .when(F.col("gold_historical_delinquency_flag") == 1, "HISTORICAL_DELINQUENCY_REVIEW")
         .when(F.col("gold_has_overdue_amount_flag") == 1, "OVERDUE_EXPOSURE_REVIEW")
         .when(
             F.col("gold_enquiries_30d") >= 6,
             "HIGH_RECENT_ENQUIRY_REVIEW"
         )
         .when(
             F.col("gold_report_age_days") > 180,
             "STALE_BUREAU_REPORT"
         )
         .otherwise("NO_BASIC_REVIEW_SIGNAL")
    )
    .withColumn(
        "gold_data_quality_status",
        F.when(F.col("gold_bureau_record_id").isNull(), "INVALID")
         .when(F.col("gold_customer_id").isNull(), "INVALID")
         .when(
             F.col("gold_credit_score").isNull() |
             (F.col("gold_credit_score") < 300) |
             (F.col("gold_credit_score") > 900),
             "INVALID"
         )
         .when(
             F.col("gold_outstanding_amount").isNull() |
             F.col("gold_overdue_amount").isNull() |
             (F.col("gold_outstanding_amount") < 0) |
             (F.col("gold_overdue_amount") < 0),
             "INVALID"
         )
         .otherwise("VALID")
    )
)

# ============================================================
# 11. GOLD METADATA
# ============================================================
df = (
    df
    .withColumn("gold_created_at", F.current_timestamp())
    .withColumn("gold_source_table", F.lit(source_table))
)

gold_df = df

# ============================================================
# 12. APPEND TO GOLD
# Skip IDs already in Gold for normal sequential reruns.
# ============================================================
if spark.catalog.tableExists(target_table):
    existing_ids = (
        spark.read.table(target_table)
        .select(F.col("bureau_record_id").alias("_existing_bureau_record_id"))
        .filter(F.col("_existing_bureau_record_id").isNotNull())
        .dropDuplicates()
    )

    gold_df = gold_df.join(
        existing_ids,
        gold_df["bureau_record_id"] == existing_ids["_existing_bureau_record_id"],
        "left_anti"
    )

gold_df.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(target_table)

print(f"Gold data appended successfully to {target_table}")

# ============================================================
# 13. VALIDATE GOLD TABLE
# ============================================================
spark.sql(f"""
SELECT
    COUNT(*) AS total_bureau_records,
    COUNT(DISTINCT bureau_record_id) AS unique_bureau_records,
    COUNT(DISTINCT customer_id) AS unique_customers,
    ROUND(AVG(gold_credit_score), 2) AS avg_credit_score,
    SUM(gold_has_outstanding_debt_flag) AS records_with_outstanding_debt,
    SUM(gold_has_overdue_amount_flag) AS records_with_overdue_amount,
    SUM(gold_historical_delinquency_flag) AS historical_delinquency_records,
    SUM(gold_written_off_account_flag) AS records_with_writeoffs,
    SUM(CASE WHEN gold_review_signal <> 'NO_BASIC_REVIEW_SIGNAL' THEN 1 ELSE 0 END)
        AS records_flagged_for_review,
    SUM(CASE WHEN gold_data_quality_status = 'INVALID' THEN 1 ELSE 0 END)
        AS invalid_records
FROM {target_table}
""").show(truncate=False)

# Score-band distribution
spark.sql(f"""
SELECT
    gold_credit_score_segment,
    COUNT(*) AS bureau_records,
    COUNT(DISTINCT customer_id) AS unique_customers,
    ROUND(AVG(gold_credit_score), 2) AS avg_credit_score,
    ROUND(AVG(gold_outstanding_amount), 2) AS avg_outstanding_amount,
    ROUND(AVG(gold_overdue_amount), 2) AS avg_overdue_amount
FROM {target_table}
GROUP BY gold_credit_score_segment
ORDER BY gold_credit_score_segment
""").show(truncate=False)

# Preview
spark.read.table(target_table).show(10, truncate=False)
